# Hemodynamic Analysis — OpenFOAM + ParaView Pipeline

This notebook computes key hemodynamic metrics from CFD simulation data (segment_test_V2 case).

**Workflow:**
1. Load ParaView Integrate Variables CSV exports
2. Inspect columns and interpret units
3. Extract physical quantities (area, velocity, pressure)
4. Compute hemodynamic metrics: P_mean, ΔP, Q, R
5. Validate mass conservation and pressure drop
6. Plot results

**Note on units:**
OpenFOAM uses *kinematic pressure* `p` [m²/s²], which equals P/ρ.  
To recover physical pressure in Pa: `P_phys = ρ × p_kinematic`  
For blood: ρ ≈ 1060 kg/m³.

**About ParaView Integrate Variables output:**  
The filter computes ∫f dA over a surface — the spatial integral of each field  
over the slice area. So:
- `Area` = ∫1 dA  [m²]  — geometric cross-sectional area
- `p`    = ∫p dA  [m⁴/s²] — integrated kinematic pressure  
- `U:i`  = ∫U_i dA [m³/s] — integrated velocity component i  

To get mean quantities, divide by Area.

## Section 1 — Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
# DATA_DIR contains the CSV files exported from ParaView
DATA_DIR = Path("../openfoam/segment_test_V2/data")

# ── Load "Integrate Variables" summary files (1-row per slice) ────────────
iv_inlet    = pd.read_csv(DATA_DIR / "inlet_integration_variable.csv")
iv_mid      = pd.read_csv(DATA_DIR / "midslice_integration_variable.csv")
iv_outlet   = pd.read_csv(DATA_DIR / "outlet_integration_variable.csv")

# ── Load per-cell raw files (many rows, one per mesh cell on the slice) ───
raw_inlet   = pd.read_csv(DATA_DIR / "inlet_integration.csv")
raw_mid     = pd.read_csv(DATA_DIR / "midslice_integration.csv")
raw_outlet  = pd.read_csv(DATA_DIR / "outlet_integration.csv")

print("Integrate-Variables summary loaded:")
print(f"  inlet   : {iv_inlet.shape}")
print(f"  midslice: {iv_mid.shape}")
print(f"  outlet  : {iv_outlet.shape}")
print()
print("Per-cell raw data loaded:")
print(f"  inlet   : {raw_inlet.shape}")
print(f"  midslice: {raw_mid.shape}")
print(f"  outlet  : {raw_outlet.shape}")

## Section 2 — Inspect Columns and Units

In [ ]:
# ── Column legend ──────────────────────────────────────────────────────────
# U:0, U:1, U:2  → integrated velocity components [m³/s]  (∫U_i dA)
# p               → integrated kinematic pressure [m⁴/s²]  (∫p_kin dA)
# Area            → cross-sectional area [m²]
# Cell Type       → VTK cell type code (ignore)

print("=== Integrate-Variables file (inlet) ===")
print(iv_inlet.to_string())
print()
print("=== Integrate-Variables file (midslice) ===")
print(iv_mid.to_string())
print()
print("=== Integrate-Variables file (outlet) ===")
print(iv_outlet.to_string())
print()
print("=== Per-cell raw file columns (inlet) ===")
print(raw_inlet.columns.tolist())
print(f"  rows: {len(raw_inlet)}")
print(raw_inlet.describe())

## Section 3 — Extract Physical Quantities

### How Integrate Variables maps to physics

| CSV column | Integral | Mean value formula | Physical meaning |
|---|---|---|---|
| `Area` | ∫ 1 dA | — | cross-sectional area [m²] |
| `U:0,1,2` | ∫ U_i dA | U_mean_i = U_i_int / Area | integrated velocity component [m³/s] |
| `p` | ∫ p_kin dA | p_mean_kin = p_int / Area | integrated kinematic pressure [m⁴/s²] → mean in m²/s² |

**Volumetric flow rate Q**:  
Since `U:0,1,2` = ∫U_i dA, the vector magnitude gives Q = ‖∫U dA‖ = ∫|U·n̂| dA ≈ Q [m³/s].  
This works when flow is approximately aligned through the slice (dominant normal component).  
For a more precise result you would need the slice normal vector to project onto it.

**Blood density**: ρ = 1060 kg/m³ (typical for human blood)  
**To convert kinematic → physical pressure**: P_phys [Pa] = ρ × p_kinematic [m²/s²]

In [ ]:
RHO = 1060.0  # blood density [kg/m³]

def extract_quantities(iv_df, label: str) -> dict:
    """
    Extract physical quantities from a single-row Integrate Variables DataFrame.

    Parameters
    ----------
    iv_df  : DataFrame with columns U:0, U:1, U:2, p, Area
    label  : human-readable name ('inlet', 'midslice', 'outlet')

    Returns
    -------
    dict with keys: label, area, U_int, Q, p_int_kin, p_mean_kin, p_mean_Pa
    """
    row = iv_df.iloc[0]

    area = row["Area"]                          # [m²]

    # Integrated velocity vector [m³/s]
    U_int = np.array([row["U:0"], row["U:1"], row["U:2"]])

    # Volumetric flow rate = magnitude of integrated velocity [m³/s]
    Q = np.linalg.norm(U_int)

    # Integrated kinematic pressure [m⁴/s²]
    p_int_kin = row["p"]

    # Mean kinematic pressure = integral / area [m²/s²]
    p_mean_kin = p_int_kin / area

    # Convert to physical pressure [Pa]
    p_mean_Pa = RHO * p_mean_kin

    return {
        "label":      label,
        "area_m2":    area,
        "U_int":      U_int,
        "Q_m3s":      Q,
        "p_int_kin":  p_int_kin,
        "p_mean_kin": p_mean_kin,
        "p_mean_Pa":  p_mean_Pa,
    }


inlet   = extract_quantities(iv_inlet,  "inlet")
midslice = extract_quantities(iv_mid,   "midslice")
outlet  = extract_quantities(iv_outlet, "outlet")

# Pretty-print
print(f"{'Location':<12} {'Area [mm²]':>12} {'Q [mL/s]':>12} {'p_mean_kin [m²/s²]':>22} {'p_mean [Pa]':>14}")
print("-" * 76)
for s in [inlet, midslice, outlet]:
    print(f"{s['label']:<12} "
          f"{s['area_m2']*1e6:>12.4f} "
          f"{s['Q_m3s']*1e6:>12.6f} "
          f"{s['p_mean_kin']:>22.6f} "
          f"{s['p_mean_Pa']:>14.4f}")

## Section 4 — Compute Hemodynamic Metrics

$$P_{mean} = \frac{\int p \, dA}{A}$$

$$\Delta P = P_{inlet} - P_{outlet}$$

$$Q = \left\| \int \mathbf{U} \, dA \right\|$$

$$R = \frac{\Delta P}{Q}$$

Note: R has units of Pa·s/m³ (= Pa/(m³/s)). For clinical comparison, 1 mmHg/(mL/s) = 133322 Pa·s/m³.

In [ ]:
# ── Mean pressures [Pa] ────────────────────────────────────────────────────
P_inlet    = inlet["p_mean_Pa"]
P_midslice = midslice["p_mean_Pa"]
P_outlet   = outlet["p_mean_Pa"]

# ── Pressure drop ΔP [Pa] ─────────────────────────────────────────────────
dP_total   = P_inlet - P_outlet        # inlet → outlet
dP_first   = P_inlet - P_midslice      # inlet → midslice
dP_second  = P_midslice - P_outlet     # midslice → outlet

# ── Flow rates [m³/s] ─────────────────────────────────────────────────────
Q_inlet    = inlet["Q_m3s"]
Q_midslice = midslice["Q_m3s"]
Q_outlet   = outlet["Q_m3s"]

# Representative Q: use inlet (most reliable boundary condition)
Q_ref = Q_inlet

# ── Hydraulic resistance R = ΔP / Q [Pa·s/m³] ────────────────────────────
R_total    = dP_total  / Q_ref
R_first    = dP_first  / Q_ref
R_second   = dP_second / Q_ref

# ── Conversion factor: Pa·s/m³ → mmHg/(mL/s) ─────────────────────────────
#   1 Pa = 1/133.322 mmHg;  1 m³/s = 1e6 mL/s
#   → 1 Pa·s/m³ = (1/133.322) mmHg / (1e6 mL/s) = 7.5e-9 mmHg·s/mL
R_factor   = 1 / 133.322e6  # Pa·s/m³ → mmHg·s/mL

print("=" * 55)
print("  HEMODYNAMIC METRICS SUMMARY")
print("=" * 55)
print(f"\nMean pressures:")
print(f"  P_inlet    = {P_inlet:.4f}  Pa  ({P_inlet/133.322:.4f} mmHg)")
print(f"  P_midslice = {P_midslice:.4f}  Pa  ({P_midslice/133.322:.4f} mmHg)")
print(f"  P_outlet   = {P_outlet:.4f}  Pa  ({P_outlet/133.322:.4f} mmHg)")

print(f"\nPressure drops:")
print(f"  ΔP (inlet→outlet)   = {dP_total:.4f}  Pa  ({dP_total/133.322:.4f} mmHg)")
print(f"  ΔP (inlet→midslice) = {dP_first:.4f}  Pa  ({dP_first/133.322:.4f} mmHg)")
print(f"  ΔP (mid→outlet)     = {dP_second:.4f}  Pa  ({dP_second/133.322:.4f} mmHg)")

print(f"\nFlow rates:")
print(f"  Q_inlet    = {Q_inlet*1e6:.6f} mL/s")
print(f"  Q_midslice = {Q_midslice*1e6:.6f} mL/s")
print(f"  Q_outlet   = {Q_outlet*1e6:.6f} mL/s")

print(f"\nHydraulic resistance R = ΔP / Q (using Q_inlet as reference):")
print(f"  R_total   = {R_total:.3e} Pa·s/m³  ({R_total*R_factor:.4f} mmHg·s/mL)")
print(f"  R_first   = {R_first:.3e} Pa·s/m³  ({R_first*R_factor:.4f} mmHg·s/mL)")
print(f"  R_second  = {R_second:.3e} Pa·s/m³  ({R_second*R_factor:.4f} mmHg·s/mL)")
print("=" * 55)

## Section 5 — Validation Checks

### Mass conservation (incompressible flow)
For incompressible steady-state flow: Q_inlet ≈ Q_outlet.  
Tolerance: deviation < 5% is acceptable; > 10% suggests a problem with mesh or slice orientation.

### Pressure monotonicity
Pressure must decrease from inlet → midslice → outlet in a pressure-driven flow.
If P_outlet > P_inlet, check boundary conditions or slice placement.

In [ ]:
PASS = "✓ PASS"
FAIL = "✗ FAIL"
WARN = "⚠ WARN"

# ── 1. Mass conservation ───────────────────────────────────────────────────
mass_error = abs(Q_outlet - Q_inlet) / Q_inlet * 100  # [%]
mass_ok    = PASS if mass_error < 5 else (WARN if mass_error < 10 else FAIL)

print("=== VALIDATION REPORT ===\n")
print(f"1. Mass conservation (incompressible steady flow)")
print(f"   Q_inlet   = {Q_inlet*1e6:.6f} mL/s")
print(f"   Q_outlet  = {Q_outlet*1e6:.6f} mL/s")
print(f"   Deviation = {mass_error:.2f}%  →  {mass_ok}")
print(f"   (acceptable: <5%; concerning: >10%)")

# ── 2. Pressure monotonicity ───────────────────────────────────────────────
p_mono = (P_inlet >= P_midslice) and (P_midslice >= P_outlet)
p_mono_str = PASS if p_mono else FAIL

print(f"\n2. Pressure decreases downstream")
print(f"   P_inlet ({P_inlet:.4f} Pa) ≥ P_midslice ({P_midslice:.4f} Pa) ≥ P_outlet ({P_outlet:.4f} Pa)")
print(f"   → {p_mono_str}")
if not p_mono:
    print("   !! Check: slice orientation, BC setup, or slice placement")

# ── 3. Positive pressure drop ─────────────────────────────────────────────
dp_ok = PASS if dP_total > 0 else FAIL
print(f"\n3. Positive total pressure drop")
print(f"   ΔP = {dP_total:.4f} Pa  →  {dp_ok}")

# ── 4. Reasonable flow rate ───────────────────────────────────────────────
# Typical ICA flow: ~2–6 mL/s; CoW segments: 0.1–2 mL/s
Q_mls = Q_inlet * 1e6
if 0.01 < Q_mls < 10:
    q_check = PASS
elif Q_mls < 0.001 or Q_mls > 100:
    q_check = FAIL
else:
    q_check = WARN
print(f"\n4. Physiological flow rate range (0.01–10 mL/s for CoW segments)")
print(f"   Q = {Q_mls:.6f} mL/s  →  {q_check}")

## Section 6 — Plots

### 6a — Pressure profile along vessel axis  
### 6b — Flow rate comparison across slices  
### 6c — Per-cell pressure and velocity distributions (from raw CSV)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Hemodynamic Analysis — segment_test_V2", fontsize=14, fontweight="bold")

locations   = ["Inlet", "Midslice", "Outlet"]
p_values_Pa = [P_inlet, P_midslice, P_outlet]
p_values_mmHg = [v / 133.322 for v in p_values_Pa]
Q_values    = [Q_inlet*1e6, Q_midslice*1e6, Q_outlet*1e6]
colors      = ["steelblue", "darkorange", "seagreen"]

# ── Plot 6a: Pressure profile ──────────────────────────────────────────────
ax = axes[0]
ax.plot(locations, p_values_mmHg, "o-", color="crimson", linewidth=2, markersize=8)
for i, (loc, v) in enumerate(zip(locations, p_values_mmHg)):
    ax.annotate(f"{v:.3f} mmHg", xy=(i, v), xytext=(0, 10),
                textcoords="offset points", ha="center", fontsize=9)
ax.set_title("Mean Pressure Profile", fontweight="bold")
ax.set_ylabel("Mean Pressure [mmHg]")
ax.set_xlabel("Location")
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))

# Annotate ΔP
ax.annotate("", xy=(2, p_values_mmHg[2]), xytext=(0, p_values_mmHg[0]),
            arrowprops=dict(arrowstyle="<->", color="gray", lw=1.5))
ax.text(2.1, (p_values_mmHg[0] + p_values_mmHg[2]) / 2,
        f"ΔP={dP_total/133.322:.4f} mmHg", va="center", fontsize=8, color="gray")

# ── Plot 6b: Flow rate comparison ─────────────────────────────────────────
ax = axes[1]
bars = ax.bar(locations, Q_values, color=colors, edgecolor="black", linewidth=0.8)
for bar, val in zip(bars, Q_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(Q_values) * 0.02,
            f"{val:.5f}", ha="center", va="bottom", fontsize=9)
ax.axhline(Q_values[0], color="crimson", linestyle="--", linewidth=1.2, label=f"Q_inlet ref")
ax.set_title("Volumetric Flow Rate Q", fontweight="bold")
ax.set_ylabel("Q [mL/s]")
ax.set_xlabel("Location")
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)

# ── Plot 6c: Per-cell pressure distributions ──────────────────────────────
ax = axes[2]
for raw, label, color in [(raw_inlet,   "Inlet",    "steelblue"),
                           (raw_mid,     "Midslice", "darkorange"),
                           (raw_outlet,  "Outlet",   "seagreen")]:
    p_kin = raw["p"]
    p_Pa  = p_kin * RHO
    p_mmHg = p_Pa / 133.322
    ax.hist(p_mmHg, bins=30, alpha=0.55, label=label, color=color, edgecolor="none")

ax.set_title("Per-Cell Pressure Distribution", fontweight="bold")
ax.set_xlabel("Pressure [mmHg]")
ax.set_ylabel("Cell count")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "hemodynamics_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to data/hemodynamics_summary.png")

In [ ]:
# ── Per-cell velocity magnitude distributions ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Per-Cell Velocity Distributions", fontsize=13, fontweight="bold")

for raw, label, color in [(raw_inlet,   "Inlet",    "steelblue"),
                           (raw_mid,     "Midslice", "darkorange"),
                           (raw_outlet,  "Outlet",   "seagreen")]:
    U_mag = np.sqrt(raw["U:0"]**2 + raw["U:1"]**2 + raw["U:2"]**2)
    axes[0].hist(U_mag * 100, bins=30, alpha=0.55, label=label, color=color)
    axes[1].hist(raw["U:0"] * 100, bins=30, alpha=0.55, label=f"{label} U:0", color=color)

axes[0].set_title("|U| distribution")
axes[0].set_xlabel("|U| [cm/s]")
axes[0].set_ylabel("Cell count")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("U:0 (x-component) distribution")
axes[1].set_xlabel("U:0 [cm/s]")
axes[1].set_ylabel("Cell count")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "velocity_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to data/velocity_distributions.png")

## Summary Table — All Results

In [ ]:
summary = pd.DataFrame({
    "Location":        ["Inlet", "Midslice", "Outlet"],
    "Area [mm²]":      [s["area_m2"] * 1e6 for s in [inlet, midslice, outlet]],
    "Q [mL/s]":        [s["Q_m3s"] * 1e6   for s in [inlet, midslice, outlet]],
    "p_mean_kin [m²/s²]": [s["p_mean_kin"] for s in [inlet, midslice, outlet]],
    "p_mean [Pa]":     [s["p_mean_Pa"]      for s in [inlet, midslice, outlet]],
    "p_mean [mmHg]":   [s["p_mean_Pa"] / 133.322 for s in [inlet, midslice, outlet]],
})

derived = pd.DataFrame({
    "Metric": [
        "ΔP_total (inlet→outlet) [Pa]",
        "ΔP_total (inlet→outlet) [mmHg]",
        "R_total [Pa·s/m³]",
        "R_total [mmHg·s/mL]",
        "Mass conservation error [%]",
    ],
    "Value": [
        f"{dP_total:.6f}",
        f"{dP_total/133.322:.6f}",
        f"{R_total:.4e}",
        f"{R_total * R_factor:.6f}",
        f"{mass_error:.2f}",
    ]
})

print("=== Per-slice summary ===")
print(summary.to_string(index=False))
print()
print("=== Derived metrics ===")
print(derived.to_string(index=False))